In [3]:
import jieba
import random
import pkuseg
import zhconv
import json


## 读取训练数据

In [4]:
all_data = []
with open("data/dev.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        one_line = json.loads(line)
        all_data.append([one_line['sentence'], one_line['label_desc'][5:]])



In [5]:
all_data[0]

['江疏影甜甜圈自拍，迷之角度竟这么好看，美吸引一切事物', 'entertainment']

In [6]:
len(all_data)

10000

In [9]:
# 清洗数据
def full2half(string):
    """
    将传入数据进行全角到半角的转换
    :param string: str 传入的字符串
    :return: str 转换完毕的字符串
    """
    rstring = ""
    for char in string:
        inside_code=ord(char)
        if inside_code == 12288:
            #全角空格直接转换
            inside_code = 32
            rstring += chr(inside_code)
        elif inside_code >= 65281 and inside_code <= 65374:
            #全角字符（除空格） 根据关系转化
            inside_code -= 65248
            rstring += chr(inside_code)
        else:
            rstring += chr(inside_code)
    return rstring


In [12]:
test_string = "abc欢迎来到万门大学ｐｙｔｈｏｎ"

In [13]:
full2half(test_string)

'abc欢迎来到万门大学python'

In [14]:
def data_cleaning(words, cleaning_parameters):
    """
    将传入数据进行数据清洗
    :param words: str 传入的字符串
    :param cleaning_parameters: list 传入的控制开关列表
    :return: str 清洗完毕的字符串
    """
    if cleaning_parameters[0]:
        words = zhconv.convert(words, 'zh-cn')
    if cleaning_parameters[1]:
        words = words.lower()
    if cleaning_parameters[2]:
        words = "".join(words.split())
    if cleaning_parameters[3]:
        words = full2half(words)
    return words


In [16]:

test_words = "aBc 歡迎来到万门大学ｐｙｔｈｏｎ"
cleaning_parameters = [True, True, True, True]
data_cleaning(test_words, cleaning_parameters)


'abc欢迎来到万门大学python'

## 数据分词

In [17]:

class Single_Tokenizer():
    def cut(self, words):
        seg_words = list(words)
        return seg_words


In [19]:
def make_tokenizer(tokenizer_name, userdict_path="", stopwords_path=""):
    """
    选取分词器
    :param tokenizer_name: str 分词器的名称
    :param userdict_path: str 自定义词典的路径
    :param stopwords_path: str 停用词表的路径
    :return: object 分词器返回
    """
    if stopwords_path != "":
        # 将停用词读出来放在stopwords这个列表中
        stopwords = [line.strip() for line in open(stopwords_path, 'r', encoding='utf-8').readlines()]
    else:
        stopwords = []
    if tokenizer_name == "pkuseg":
        if userdict_path != "":
            pku = pkuseg.pkuseg(user_dict=userdict_path)
        else:
            pku = pkuseg.pkuseg()
        return pku, stopwords
    elif tokenizer_name == "single":
        single_tokenizer = Single_Tokenizer()
        return single_tokenizer, stopwords
    else:
        # 默认使用jieba分词
        if userdict_path != "":
            jieba.load_userdict(userdict_path)
        return jieba, stopwords


In [20]:
def word_seg(all_data, tokenizer, stopwords, cleaning_parameters):
    """
    数据分词，分词前进行数据清洗
    :param all_data: list 原始数据
    :param tokenizer: object 分词器
    :param stopwords: list 停用词列表
    :param cleaning_parameters: list 传入的控制开关列表
    :return: list 分完词之后的数据
    """
    segmented_all_data = []
    for sentences, tag in all_data:
        sentences = data_cleaning(sentences, cleaning_parameters)
        seg_list = tokenizer.cut(sentences)
        seg_list = [i for i in seg_list if i not in stopwords]
        segmented_all_data.append([seg_list, tag])
    return segmented_all_data


In [ ]:

tokenizer_name = "pkuseg"
userdict_path = "./userdict/userdict.txt"
stopwords_path = "./stopwords/stopwords.txt"
tokenizer, stopwords = make_tokenizer(tokenizer_name, userdict_path, stopwords_path)


In [24]:
cleaning_parameters = [True, True, True, True]
segmented_all_data = word_seg(all_data[:50], tokenizer, stopwords, cleaning_parameters)
print(segmented_all_data[:5])


[[['江', '疏影', '甜甜圈', '自拍', '迷之', '角度', '竟', '这么', '好看', '美', '吸引', '一切', '事物'], 'entertainment'], [['以色列', '大规模', '空袭', '开始', '!', '伊朗', '多', '个', '军事', '目标', '遭遇', '打击', '誓言', '对等', '反击'], 'military'], [['出栏', '一头', '猪', '亏损', '300', '元', '究竟', '谁', '能', '笑到', '最后', '!'], 'finance'], [['以前', '很', '火', '的', '巴铁', '为何', '现在', '只字不提'], 'tech'], [['作为', '一', '名', '酒店', '从业', '人员', '你', '经历', '过', '房客', '哪些', '特别', '没有', '素质', '的', '行为'], 'travel']]


## 生成word2id, tag2id

In [25]:

def make_map_dict(segmented_all_data):
    """
    制作词到ID的映射，标签到ID的映射
    :param segmented_all_data: list 分完词之后的数据
    :return: dict 词到ID的映射及标签到ID的映射
    """
    all_tag = []
    all_words = []
    all_words.append('<PAD>')
    all_words.append('<UNK>')
    for seg_list, tag in segmented_all_data:
        for word in seg_list:
            if word not in all_words:
                all_words.append(word)
        if tag not in all_tag:
            all_tag.append(tag)
    word2id = {all_words[i]: i for i in range(len(all_words))}
    id2word = {v: k for k, v in word2id.items()}
    tag2id = {all_tag[i]: i for i in range(len(all_tag))}
    id2tag = {v: k for k, v in tag2id.items()}
    return word2id, id2word, tag2id, id2tag


In [26]:

word2id, id2word, tag2id, id2tag = make_map_dict(segmented_all_data)
print(word2id)


{'<PAD>': 0, '<UNK>': 1, '江': 2, '疏影': 3, '甜甜圈': 4, '自拍': 5, '迷之': 6, '角度': 7, '竟': 8, '这么': 9, '好看': 10, '美': 11, '吸引': 12, '一切': 13, '事物': 14, '以色列': 15, '大规模': 16, '空袭': 17, '开始': 18, '!': 19, '伊朗': 20, '多': 21, '个': 22, '军事': 23, '目标': 24, '遭遇': 25, '打击': 26, '誓言': 27, '对等': 28, '反击': 29, '出栏': 30, '一头': 31, '猪': 32, '亏损': 33, '300': 34, '元': 35, '究竟': 36, '谁': 37, '能': 38, '笑到': 39, '最后': 40, '以前': 41, '很': 42, '火': 43, '的': 44, '巴铁': 45, '为何': 46, '现在': 47, '只字不提': 48, '作为': 49, '一': 50, '名': 51, '酒店': 52, '从业': 53, '人员': 54, '你': 55, '经历': 56, '过': 57, '房客': 58, '哪些': 59, '特别': 60, '没有': 61, '素质': 62, '行为': 63, '走': 64, '进': 65, '荀子': 66, '世界': 67, '触摸': 68, '二千': 69, '年前': 70, '心灵': 71, '温度': 72, '图解': 73, '全': 74, '要素': 75, '领域': 76, '高': 77, '效益': 78, '天津': 79, '智能': 80, '科技': 81, '军民': 82, '融合': 83, '发展': 84, '区块链': 85, '投资': 86, '心得': 87, '做到': 88, '就': 89, '不': 90, '会': 91, '亏': 92, '钱': 93, '你家': 94, '拆迁': 95, '要': 96, '还是': 97, '房': 98, '答案': 99, '一目了然': 100, '军嫂': 101, 

## 数据切分

In [27]:
random.seed(1)
random.shuffle(segmented_all_data)
data_len = len(segmented_all_data)
train_ratio = int(data_len * 0.8)
train_data = segmented_all_data[:train_ratio]
test_data = segmented_all_data[train_ratio:]



In [31]:
len(train_data)
data_len

50